# Giai đoạn 2: Chọn Tập Đại diện & Khám phá Từng Doanh nghiệp (Hồi 2 — Bằng chứng vi mô)

## 00. Giới thiệu giai đoạn

- **Tên giai đoạn:** Giai đoạn 2 — Chọn tập đại diện & Khám phá từng doanh nghiệp (`02_business_eda.ipynb`).
- **Vai trò trong câu chuyện dữ liệu (Hồi 2 — *Bằng chứng vi mô*):** Từ bức tranh tổng thể 14 doanh nghiệp ở Hồi 1, lựa chọn các doanh nghiệp tiêu biểu (phủ các nhóm ngành và mức độ số hóa), làm sạch theo `rulebook.md` và Tidy Data, kiểm tra chất lượng & ngoại lệ IQR, rồi đi sâu khám phá mẫu hình vận hành nội tại của từng doanh nghiệp.
- **Mục tiêu:**
  1. Lựa chọn tập doanh nghiệp đại diện trong 14 doanh nghiệp `public_test` với tiêu chí rõ ràng, có căn cứ.
  2. Triển khai bộ hàm xử lý và tính chỉ số tuân thủ tuyệt đối [`data/public_test/rulebook.md`](../data/public_test/rulebook.md) (loại đơn hủy/trả — kể cả trả một phần, quy theo ngày đặt, doanh thu dòng hàng, quy đổi đơn vị cơ bản, giải mã `Ghi tắt`, chuẩn Tidy Data).
  3. Kiểm tra chất lượng dữ liệu (giá trị thiếu, trùng lặp, tính hợp lệ, ngoại lệ theo quy tắc IQR đối chiếu với dữ liệu gốc).
  4. Khám phá và tinh chỉnh 1–2 biểu đồ Giải thích đắt giá cho mỗi doanh nghiệp đại diện (Hồi 2).
- **Phạm vi dữ liệu:** Chỉ đọc từ `public_test` (`dev_001`–`dev_014`, `rulebook.md`, `calendar.csv`).
- **Đầu ra mong đợi (Cột mốc M2 & M3 phần Hồi 2) & Nơi lưu trữ:**
  - Thư mục gốc phase: `artifacts/guided_eda/phase2_business_eda/`
  - Dữ liệu Tidy sạch (`PHASE2_TIDY_DIR`): `tidy_data/<business_id>_tidy.csv` (bàn giao cho Giai đoạn 3).
  - Bảng kết quả (`PHASE2_TABLES_DIR`): `tables/representative_selection.csv`, `tables/quality_and_iqr_report.csv`, `tables/business_findings_summary.csv`.
  - Biểu đồ Hồi 2 (`PHASE2_FIGURES_DIR`): `figures/`.
- **Các giới hạn cần tuân thủ:**
  - Tuyệt đối không sửa dữ liệu gốc; không mặc định mọi ngoại lệ IQR là lỗi nhập liệu.
  - Không dùng dữ liệu `train` hoặc `questions.jsonl`; không suy diễn quan hệ nhân quả.


## 01. Thiết lập môi trường


In [ ]:
import sys
from pathlib import Path

# Tự động xác định thư mục gốc dự án và thêm src vào sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "common").is_dir():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src" / "common").is_dir():
            PROJECT_ROOT = parent
            break

SCRIPT_ROOT = PROJECT_ROOT / "src"
if not SCRIPT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục mã nguồn EDA tại: {SCRIPT_ROOT}"
    )

if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

print(f"Thư mục gốc dự án : {PROJECT_ROOT}")
print(f"Thư mục script EDA : {SCRIPT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import có chọn lọc các tiện ích từ src/
from common.config import load_config
from common.paths import (
    DatasetContext,
    assert_not_in_raw,
    assert_safe_output,
    is_benchmark_file,
)
from common.provenance import fingerprint_file
from readers.markdown_reader import read_business_readme, read_rulebook
from readers.csv_reader import inspect_csv, read_all_csv_rows
from readers.excel_reader import list_sheets, read_sheet_rows
from readers.sqlite_reader import inspect_sqlite, query_readonly
from readers.table_reader import LoadedTable, load_table
from inventory.file_manifest import build_file_manifest
from rulebook.rule_registry import compile_rule_registry
from business.unit_mapping import validate_unit_mapping
from quality.missingness import classify_null_kind, profile_missingness
from quality.duplicates import profile_duplicates
from eda_statistics.robust_stats import compute_robust_dispersion, iqr_outlier_masks
from schema.type_inference import parse_date, parse_datetime, parse_number

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Thiết lập giao diện biểu đồ cơ bản (hỗ trợ hiển thị inline trong Jupyter)
# Lưu ý: Không import visualization.figure_theme vì module đó ép backend Agg.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titlelocation": "left",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.55,
    "axes.unicode_minus": False,
    "figure.figsize": (10, 5.5),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Bảng màu gợi ý theo nguyên tắc "Bắt đầu từ màu xám" (Schwabish)
COLOR_GRAY = "#9AA5B1"
COLOR_GRAY_DARK = "#52606D"
COLOR_ACCENT_BLUE = "#1769AA"
COLOR_ACCENT_TEAL = "#087B5A"
COLOR_ACCENT_AMBER = "#A36516"
COLOR_ACCENT_RED = "#B32336"


## 02. Đọc dữ liệu, Khởi tạo thư mục đầu ra & Hợp đồng đầu vào

Notebook này chạy độc lập về mặt kernel so với Notebook 01. Phần dưới đây nạp sẵn cấu hình `public_test`, `rulebook.md`, khởi tạo (`mkdir`) các thư mục đầu ra của Giai đoạn 2 và kiểm tra **Hợp đồng đầu vào (Input Contract)** từ Giai đoạn 1.


In [ ]:
config = load_config(workspace_root=PROJECT_ROOT)
context: DatasetContext = config.context_for("public_test")
assert context.read_only is True
assert not is_benchmark_file(context.rulebook_path)

business_ids = context.business_ids()
rule_registry = compile_rule_registry(
    context.rulebook_path,
    split="public_test",
    rulebook_hash=fingerprint_file(context.rulebook_path),
)

# Khởi tạo (mkdir) thư mục lưu trữ đầu ra cho Giai đoạn 2 (nằm ngoài data/ và raw_data/)
GUIDED_ROOT = config.output_root / "guided_eda"
PHASE2_ROOT = GUIDED_ROOT / "phase2_business_eda"
PHASE2_TIDY_DIR = PHASE2_ROOT / "tidy_data"
PHASE2_TABLES_DIR = PHASE2_ROOT / "tables"
PHASE2_FIGURES_DIR = PHASE2_ROOT / "figures"

assert_not_in_raw(config.raw_root, GUIDED_ROOT)
for target_dir in (PHASE2_ROOT, PHASE2_TIDY_DIR, PHASE2_TABLES_DIR, PHASE2_FIGURES_DIR):
    assert_safe_output(config.output_root, target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

print(f"Đã kết nối public_test : {context.root} ({len(business_ids)} doanh nghiệp)")
print(f"Đã nạp rulebook.md     : {len(rule_registry.rules)} quy tắc thuộc {len(rule_registry.sections)} mục")
print(f"Thư mục Tidy Data      : {PHASE2_TIDY_DIR}")
print(f"Thư mục bảng Phase 2   : {PHASE2_TABLES_DIR}")
print(f"Thư mục hình Phase 2   : {PHASE2_FIGURES_DIR}")


In [ ]:
# ==============================================================================
# HỢP ĐỒNG ĐẦU VÀO TỪ GIAI ĐOẠN 1 (Notebook 01 -> Notebook 02)
# ==============================================================================
# Đường dẫn chuẩn tới bảng danh mục 14 doanh nghiệp xuất từ NB01 (có thể trỏ file khác nếu cần)
DEFAULT_PHASE1_CATALOG_PATH = GUIDED_ROOT / "phase1_dataset_discovery" / "tables" / "catalog_14.csv"
PHASE1_CATALOG_PATH: Path | None = DEFAULT_PHASE1_CATALOG_PATH

# Tùy chọn: Hoặc dán danh sách bản ghi tổng hợp từ NB01 vào đây
PHASE1_CATALOG_RECORDS: list[dict] = []

# Danh sách doanh nghiệp đại diện sẽ chốt ở T2.1 (để trống nếu chưa chọn)
SELECTED_REPRESENTATIVE_BUSINESSES: list[str] = []


def check_phase2_input_contract(
    catalog_path: Path | None = PHASE1_CATALOG_PATH,
    catalog_records: list[dict] = PHASE1_CATALOG_RECORDS,
    selected_businesses: list[str] = SELECTED_REPRESENTATIVE_BUSINESSES,
) -> tuple[pd.DataFrame | None, dict[str, str]]:
    status_report: dict[str, str] = {}
    catalog_df: pd.DataFrame | None = None

    if catalog_path is not None:
        resolved = Path(catalog_path)
        if not resolved.is_file():
            if resolved == DEFAULT_PHASE1_CATALOG_PATH:
                status_report["catalog_14"] = (
                    f"[CHỜ BỔ SUNG] Chưa có tệp bàn giao từ Giai đoạn 1 tại: {resolved}"
                )
            else:
                status_report["catalog_14"] = f"[CẢNH BÁO] Đường dẫn không tồn tại: {resolved}"
        elif resolved.suffix.lower() == ".csv":
            catalog_df = pd.read_csv(resolved)
            status_report["catalog_14"] = f"[ĐÃ SẴN SÀNG] Nạp từ tệp ({len(catalog_df)} dòng): {resolved.name}"
        elif resolved.suffix.lower() == ".json":
            catalog_df = pd.read_json(resolved)
            status_report["catalog_14"] = f"[ĐÃ SẴN SÀNG] Nạp từ tệp ({len(catalog_df)} dòng): {resolved.name}"
        else:
            status_report["catalog_14"] = f"[CẢNH BÁO] Định dạng chưa hỗ trợ: {resolved.suffix}"
    elif catalog_records:
        catalog_df = pd.DataFrame(catalog_records)
        status_report["catalog_14"] = f"[ĐÃ SẴN SÀNG] Nhập trực tiếp ({len(catalog_df)} dòng)"
    else:
        status_report["catalog_14"] = "[CHỜ BỔ SUNG] Chưa có dữ liệu danh mục từ Giai đoạn 1 (Notebook 01)"

    if selected_businesses:
        invalid = [b for b in selected_businesses if b not in business_ids]
        if invalid:
            status_report["representatives"] = f"[CẢNH BÁO] Mã doanh nghiệp không hợp lệ: {invalid}"
        else:
            status_report["representatives"] = (
                f"[ĐÃ SẴN SÀNG] Đã chọn {len(selected_businesses)} doanh nghiệp: {selected_businesses}"
            )
    else:
        status_report["representatives"] = (
            "[CHỜ BỔ SUNG] Chờ xác nhận tại T2.1 (vẫn có thể thử nghiệm độc lập trên public_test)"
        )

    for key, msg in status_report.items():
        print(f"[Hợp đồng đầu vào - {key}]: {msg}")
    return catalog_df, status_report


phase1_catalog_df, phase2_contract_status = check_phase2_input_contract()


## 03. Tổng quan dữ liệu cơ bản


In [ ]:
# Hiển thị cấu trúc các mục quy tắc trong rulebook.md để tham chiếu nhanh
rule_sections_df = pd.DataFrame([
    {
        "section": sec,
        "rule_count": sum(1 for r in rule_registry.rules if r.section == sec),
    }
    for sec in rule_registry.sections
])
display(rule_sections_df)

# Minh họa cách xem danh sách tệp của 1 doanh nghiệp mẫu
sample_biz_id = business_ids[0]
sample_biz_path = context.business_root_path(sample_biz_id)
print(f"Tệp trong doanh nghiệp mẫu {sample_biz_id}: {[p.name for p in sorted(sample_biz_path.iterdir()) if p.is_file()]}")


## 04. Các tác vụ khám phá dữ liệu (EDA Tasks)


## Tác vụ T2.1 — Chọn tập doanh nghiệp đại diện từ 14 doanh nghiệp `public_test`

### Mục tiêu
Dựa trên kết quả kiểm kê Giai đoạn 1 (hoặc khảo sát nhanh nếu chưa có bàn giao), lựa chọn một tập doanh nghiệp đại diện phủ đa dạng ngành nghề, hình thức ghi nhận và cấu trúc đặc thù để phân tích sâu.

### Câu hỏi khám phá
- Tiêu chí chọn mẫu đại diện hợp lý là gì (độ phủ ngành nghề, mức độ số hóa, độ giàu thuộc tính, độ dài/liên tục thời gian)?
- Mỗi nhóm hình thức ghi nhận (Excel thủ công, phần mềm POS, CSV nhiều file, SQLite) nên có ít nhất một đại diện nào?
- Những doanh nghiệp không được chọn làm đại diện chính đóng vai trò gì trong Hồi 1 và Hồi 3?

### Các bước thực hiện

**Bước 1 — Đối chiếu bảng kiểm kê 14 doanh nghiệp theo ngành & mức độ số hóa**
Nếu `phase1_catalog_df` chưa có, tự lập bảng tóm tắt nhanh ngành nghề/định dạng từ `README.md` của 14 doanh nghiệp để không bị chặn.

**Bước 2 — Xác định bộ tiêu chí chọn mẫu đại diện**
Định nghĩa rõ 3–5 tiêu chí chấm điểm/sàng lọc (ví dụ: tính đa dạng ngành, mức độ chi tiết dòng hàng, dữ liệu đặc thù như lô/ca/sửa xe/điện thoại, độ phủ thời gian).

**Bước 3 — Đánh giá độ giàu thông tin và chất lượng nguồn của các ứng viên**
Kiểm tra nhanh các bảng phụ trợ của từng ứng viên (danh mục hàng hóa, khách hàng, lô/hạn dùng, ghi tắt, ca làm việc).

**Bước 4 — Chốt danh sách `SELECTED_REPRESENTATIVE_BUSINESSES` và lưu `representative_selection.csv`**
Cập nhật biến `SELECTED_REPRESENTATIVE_BUSINESSES` và xuất bảng lý do lựa chọn ra `PHASE2_TABLES_DIR / 'representative_selection.csv'`.

### Gợi ý phương pháp & Lưu ý
- Không chọn ngẫu nhiên hoặc chỉ chọn theo cảm tính; cần giải thích rõ vì sao tập đại diện phản ánh được sự đa dạng của 14 doanh nghiệp `public_test`.
- Cân nhắc đưa vào ít nhất một trường hợp đặc thù về mặt nghiệp vụ hoặc định dạng nếu tiêu chí đa dạng yêu cầu.

### Triển khai


In [ ]:
# TODO (T2.1 — Bước 1: Đối chiếu bảng kiểm kê 14 doanh nghiệp theo ngành & mức độ số hóa)
# TODO: Kiểm tra phase1_catalog_df hoặc lập bảng tóm tắt nhanh nếu trạng thái đang [CHỜ BỔ SUNG]


In [ ]:
# TODO (T2.1 — Bước 2: Xác định bộ tiêu chí chọn mẫu đại diện)
# TODO: Ghi mã hóa các tiêu chí đánh giá ứng viên đại diện


In [ ]:
# TODO (T2.1 — Bước 3: Đánh giá độ giàu thông tin và chất lượng nguồn của các ứng viên)
# TODO: Đánh giá và so sánh các doanh nghiệp theo tiêu chí vừa đặt ra


In [ ]:
# TODO (T2.1 — Bước 4: Chốt danh sách `SELECTED_REPRESENTATIVE_BUSINESSES` và lưu `representative_selection.csv`)
# TODO: Gán danh sách mã doanh nghiệp được chọn vào SELECTED_REPRESENTATIVE_BUSINESSES
# và lưu bảng giải trình ra PHASE2_TABLES_DIR / 'representative_selection.csv'


### Trực quan hóa


In [ ]:
# TODO (T2.1 — Trực quan hóa)
# Có thể vẽ biểu đồ ma trận phân bố 14 doanh nghiệp theo (Ngành nghề × Hình thức ghi nhận / Mức độ số hóa) và tô màu nhấn những doanh nghiệp được chọn làm đại diện.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T2.1):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Danh sách tập doanh nghiệp đại diện đã chốt và tiêu chí lựa chọn:** ...
- **Tính đại diện về ngành nghề, mức độ số hóa và độ phủ thời gian của tập mẫu này:** ...


## Tác vụ T2.2a — Viết hàm đọc dữ liệu & tính chỉ số chuẩn `rulebook.md`

### Mục tiêu
Xây dựng các hàm xử lý dùng chung (Pure Python / pandas) cho tập đại diện để tính doanh thu, số đơn, tỷ lệ hủy/trả đúng tuyệt đối theo các quy tắc mặc định trong `data/public_test/rulebook.md`.

### Câu hỏi khám phá
- Các quy tắc nào trong `rulebook.md` ảnh hưởng trực tiếp tới việc lọc đơn, xác định ngày đặt và tính tiền đơn hàng?
- Nếu một đơn hàng có trạng thái trả hàng (kể cả trả một phần) hoặc dòng trạng thái để trống ở dòng sau của cùng một đơn, xử lý thế nào?
- Sự khác biệt giữa *Doanh thu một đơn* (tổng thành tiền dòng hàng) với *Tổng khách phải trả* (trừ giảm giá đơn, cộng ship) là gì?

### Các bước thực hiện

**Bước 1 — Rà soát các quy tắc cốt lõi trong `rulebook.md`**
Đọc kỹ Mục 1 (Đơn hàng & doanh thu), Mục 2 (Thời gian), Mục 6 (Theo ngành) và Mục 8 (Cách ghi trong file) từ `rule_registry` hoặc văn bản `rulebook.md`.

**Bước 2 — Ánh xạ cột trạng thái & lọc đơn hủy/trả hàng (kể cả trả một phần)**
Viết logic chuẩn hóa cột trạng thái theo quy ước từng doanh nghiệp (hoàn thành / đang giao / huỷ / trả hàng) và loại toàn bộ đơn bị hủy hoặc có trả hàng khi tính doanh thu & số đơn.

**Bước 3 — Chuẩn hóa ngày đặt hàng làm mốc quy thuộc kỳ**
Xác định chính xác cột ngày đặt hàng (tránh nhầm với ngày giao hay ngày thu tiền) và chuyển sang kiểu `datetime`/`date` thống nhất.

**Bước 4 — Tính giá trị dòng hàng và giá trị đơn hàng chuẩn `rulebook.md`**
Tính doanh thu theo tổng thành tiền các dòng hàng (đã trừ giảm giá dòng, không trừ giảm giá cả đơn, không cộng VAT/phí ship); tách riêng các dòng `Phí phục vụ` nếu có.

**Bước 5 — Kiểm tra đối chiếu kết quả với bảng gốc**
Chọn 3–5 mã đơn hàng cụ thể (gồm 1 đơn thường, 1 đơn nhiều dòng, 1 đơn hủy/trả) để kiểm toán thủ công (audit) xem hàm tính có khớp 100% với tệp gốc không.

### Gợi ý phương pháp & Lưu ý
- Đọc kỹ Mục 8 `rulebook.md`: *Ô để trống ở các dòng sau của cùng một nhóm dòng giống ô đó ở dòng đầu nhóm (forward-fill theo nhóm đơn)* và *Hai dòng giống hệt nhau của cùng một đơn là lỗi xuất file, tính một lần; hai đơn khác nhau vẫn tính cả hai*.
- Chú ý các luật đặc thù theo ngành (Mục 6 `rulebook.md`): ví dụ món + topping ở cà phê, phiếu bảo hành ở sửa xe, tiền thu máy cũ / trả góp ở cửa hàng điện thoại.
- Có thể tận dụng `read_all_csv_rows`, `read_sheet_rows`, `query_readonly` để nạp bảng sạch trước khi đưa vào DataFrame.

### Triển khai


In [ ]:
# TODO (T2.2a — Bước 1: Rà soát các quy tắc cốt lõi trong `rulebook.md`)
# TODO: Lọc và in các quy tắc liên quan đến đơn hàng, doanh thu, trạng thái và ngành của tập đại diện


In [ ]:
# TODO (T2.2a — Bước 2: Ánh xạ cột trạng thái & lọc đơn hủy/trả hàng (kể cả trả một phần))
# TODO: Viết hàm chuẩn hóa trạng thái đơn và xử lý các dòng để trống trạng thái trong cùng một mã đơn (Mục 8 rulebook)


In [ ]:
# TODO (T2.2a — Bước 3: Chuẩn hóa ngày đặt hàng làm mốc quy thuộc kỳ)
# TODO: Viết hàm chuẩn hóa ngày đặt hàng cho từng doanh nghiệp đại diện


In [ ]:
# TODO (T2.2a — Bước 4: Tính giá trị dòng hàng và giá trị đơn hàng chuẩn `rulebook.md`)
# TODO: Viết hàm tính doanh thu cấp dòng và cấp đơn hàng đúng chuẩn rulebook.md


In [ ]:
# TODO (T2.2a — Bước 5: Kiểm tra đối chiếu kết quả với bảng gốc)
# TODO: Thực hiện đối chiếu (audit) 3–5 mã đơn cụ thể giữa bảng gốc và hàm vừa viết


### Kết quả & Nhận định

> **TODO (T2.2a):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Tóm tắt các quy tắc `rulebook.md` đặc thù áp dụng cho từng doanh nghiệp trong tập đại diện:** ...
- **Kết quả đối chiếu (audit) trên các mã đơn kiểm thử:** ...


## Tác vụ T2.2b — Chuẩn hóa đặc thù theo doanh nghiệp (Quy đổi đơn vị, `Ghi tắt`, Tidy Data)

### Mục tiêu
Chuẩn hóa các quy ước riêng của từng doanh nghiệp (quy đổi về đơn vị cơ bản, giải mã `Ghi tắt`, tách năm từ tên sheet nếu ngày thiếu năm) và xuất dữ liệu dạng Tidy Data ra `PHASE2_TIDY_DIR`.

### Câu hỏi khám phá
- Doanh nghiệp nào có bảng quy đổi đơn vị tính (thùng/lốc/hộp/vỉ -> đơn vị cơ bản) hoặc ghi quy ước này trong `README.md`?
- Doanh nghiệp nào sử dụng bảng `Ghi tắt` (tên hàng, loại xe, mã ghi chú) cần giải mã trước khi xếp hạng hoặc nhóm?
- Những bảng nào đang ở dạng rộng (wide), chia nhiều sheet/tệp theo tháng hoặc gộp nhiều biến trong một cột cần đưa về dạng dài (Tidy Data)?

### Các bước thực hiện

**Bước 1 — Khám phá sự đa dạng đơn vị tính và bảng quy đổi**
Kiểm tra cột `Đơn vị tính`, `Số lượng quy đổi` (nếu có sẵn) và các bảng/quy ước quy đổi đơn vị của từng doanh nghiệp đại diện.

**Bước 2 — Xây dựng logic quy đổi số lượng về đơn vị cơ bản**
Viết hàm ánh xạ `(mặt hàng, đơn vị tính) -> hệ số quy đổi` để quy toàn bộ số lượng bán về đơn vị cơ bản trước khi xếp hạng.

**Bước 3 — Giải mã bảng `Ghi tắt` và chuẩn hóa danh mục hàng/nhóm hàng**
Nạp bảng `Ghi tắt` (nếu có) và áp dụng từ điển ánh xạ chuẩn (giữ nguyên tên gốc ở một cột riêng để đối chiếu provenance).

**Bước 4 — Chuyển đổi sang Tidy Data và xuất ra `PHASE2_TIDY_DIR`**
Gộp các sheet theo tháng (nếu cùng cấu trúc, bổ sung năm từ tên sheet nếu ngày thiếu năm), tổ chức bảng dạng dài và lưu `<business_id>_tidy.csv` vào `PHASE2_TIDY_DIR`.

**Bước 5 — Kiểm tra bảo toàn tổng số liệu trước và sau chuẩn hóa**
Đối chiếu tổng số dòng hợp lệ, tổng doanh thu và tổng số lượng trước và sau bước chuẩn hóa/Tidy để chứng minh không mất mát hay nhân đôi bản ghi (fanout).

### Gợi ý phương pháp & Lưu ý
- Theo `rulebook.md` (Mục 3): khi xếp hạng *bán chạy*, phải quy về đơn vị cơ bản; không cộng số lượng khác đơn vị gốc khi chưa quy đổi.
- Khi gộp các bảng (ví dụ dòng hàng với danh mục hàng hóa để lấy `Nhóm hàng` hoặc `Giá vốn`), luôn kiểm tra số dòng trước và sau `merge` để tránh bùng nổ Cartesian (`fanout > 1`).
- Lưu từng bảng sạch `<business_id>_tidy.csv` trong `PHASE2_TIDY_DIR` giúp Notebook 03 tự động nạp được ở bước đối chiếu.

### Triển khai


In [ ]:
# TODO (T2.2b — Bước 1: Khám phá sự đa dạng đơn vị tính và bảng quy đổi)
# TODO: Liệt kê các đơn vị tính xuất hiện và kiểm tra tính nhất quán bằng validate_unit_mapping nếu có bảng quy đổi


In [ ]:
# TODO (T2.2b — Bước 2: Xây dựng logic quy đổi số lượng về đơn vị cơ bản)
# TODO: Triển khai hàm quy đổi số lượng về đơn vị cơ bản cho các doanh nghiệp cần thiết


In [ ]:
# TODO (T2.2b — Bước 3: Giải mã bảng `Ghi tắt` và chuẩn hóa danh mục hàng/nhóm hàng)
# TODO: Giải mã Ghi tắt và chuẩn hóa tên mặt hàng / nhóm hàng (không tự ý gộp nhóm khác tên theo rulebook)


In [ ]:
# TODO (T2.2b — Bước 4: Chuyển đổi sang Tidy Data và xuất ra `PHASE2_TIDY_DIR`)
# TODO: Chuẩn hóa dữ liệu dạng Tidy vào tidy_tables[biz_id] và lưu CSV vào PHASE2_TIDY_DIR
tidy_tables = {}
# for biz_id, df in tidy_tables.items():
#     df.to_csv(PHASE2_TIDY_DIR / f'{biz_id}_tidy.csv', index=False)


In [ ]:
# TODO (T2.2b — Bước 5: Kiểm tra bảo toàn tổng số liệu trước và sau chuẩn hóa)
# TODO: Viết kiểm tra đối chiếu tổng trước và sau chuẩn hóa (reconciliation check)


### Kết quả & Nhận định

> **TODO (T2.2b):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Các chuyển đổi chuẩn hóa đã thực hiện cho từng doanh nghiệp đại diện:** ...
- **Kết quả kiểm tra bảo toàn số dòng và tổng tiền sau chuẩn hóa Tidy Data:** ...


## Tác vụ T2.2c — Kiểm tra chất lượng dữ liệu tập đại diện (Thiếu dữ liệu, Trùng lặp, Ngoại lệ IQR)

### Mục tiêu
Thực hiện vệ sinh và đánh giá chất lượng dữ liệu toàn diện cho tập đại diện theo Chương 2 §2.3: thống kê thiếu dữ liệu, kiểm tra trùng lặp, kiểm tra dòng tổng cộng/hợp lệ và phát hiện ngoại lệ bằng quy tắc IQR.

### Câu hỏi khám phá
- Những cột nào bị thiếu dữ liệu? Đâu là thiếu có cấu trúc (ví dụ: khách vãng lai không có mã khách, dòng gộp ô) và đâu là khuyết thông tin thật?
- Có bản ghi trùng lặp toàn dòng hoặc trùng khóa nghiệp vụ, hoặc dòng tổng kết/subtotal xen lẫn trong bảng dữ liệu không?
- Những giá trị số nào nằm ngoài hàng rào IQR (`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`)? Khi đối chiếu với tệp gốc, đó là lỗi nhập liệu hay giao dịch thực hợp lệ (ví dụ: đơn mua sỉ)?

### Các bước thực hiện

**Bước 1 — Thống kê tỷ lệ thiếu dữ liệu & phân loại bản chất thiếu**
Tính số lượng và tỷ lệ thiếu theo cột cho từng bảng Tidy; sử dụng `profile_missingness` / `classify_null_kind` hoặc `isna()` để phân biệt thiếu cấu trúc với thiếu ngẫu nhiên.

**Bước 2 — Kiểm tra trùng lặp toàn dòng & theo khóa nghiệp vụ**
Kiểm tra trùng lặp toàn dòng (chú ý quy tắc Mục 8 `rulebook.md`: trùng trong cùng 1 đơn vs trùng ở 2 đơn khác nhau) và trùng mã khóa chính.

**Bước 3 — Kiểm tra tính hợp lệ, giá trị âm/0 và dòng tổng cộng (subtotal/summary rows)**
Rà soát các dòng có giá trị âm (trả hàng, giảm giá), giá trị 0, sai định dạng số/ngày, hoặc dòng tổng kết cuối file.

**Bước 4 — Phát hiện ngoại lệ bằng quy tắc IQR (1.5 × IQR)**
Với các cột đo lường hợp lệ (số lượng, đơn giá, giá trị đơn hàng — KHÔNG chạy IQR trên cột mã/ID), tính `Q1`, `Q3`, `IQR`, ngưỡng `[L, U]` bằng `compute_robust_dispersion` / `iqr_outlier_masks` hoặc pandas.

**Bước 5 — Đối chiếu ngoại lệ IQR với dữ liệu nguồn & lưu `quality_and_iqr_report.csv`**
Trích xuất các bản ghi vượt ngưỡng IQR để kiểm tra ngữ cảnh gốc, quyết định giữ hay xử lý và xuất báo cáo ra `PHASE2_TABLES_DIR / 'quality_and_iqr_report.csv'`.

### Gợi ý phương pháp & Lưu ý
- Tuyệt đối **không tự động xóa (drop)** các giá trị ngoại lệ chỉ vì chúng nằm ngoài `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
- Khi `IQR == 0` (ví dụ hầu hết đơn đều mua số lượng = 1), quy tắc `1.5*IQR` bị suy biến (`degenerate_iqr=True`); cần ghi nhận rõ thay vì gắn cờ sai toàn bộ các đơn mua số lượng 2 trở lên.

### Triển khai


In [ ]:
# TODO (T2.2c — Bước 1: Thống kê tỷ lệ thiếu dữ liệu & phân loại bản chất thiếu)
# TODO: Thống kê tỷ lệ thiếu dữ liệu theo cột và phân loại (thiếu cấu trúc vs khuyết dữ liệu)


In [ ]:
# TODO (T2.2c — Bước 2: Kiểm tra trùng lặp toàn dòng & theo khóa nghiệp vụ)
# TODO: Kiểm tra trùng lặp bằng pandas hoặc profile_duplicates(...)


In [ ]:
# TODO (T2.2c — Bước 3: Kiểm tra tính hợp lệ, giá trị âm/0 và dòng tổng cộng (subtotal/summary rows))
# TODO: Kiểm tra các bản ghi có số lượng <= 0, đơn giá <= 0, lỗi parse hoặc dòng tổng cộng


In [ ]:
# TODO (T2.2c — Bước 4: Phát hiện ngoại lệ bằng quy tắc IQR (1.5 × IQR))
# TODO: Tính Q1, Q3, IQR và đếm số lượng ngoại lệ trên/dưới cho từng chỉ số đo lường


In [ ]:
# TODO (T2.2c — Bước 5: Đối chiếu ngoại lệ IQR với dữ liệu nguồn & lưu `quality_and_iqr_report.csv`)
# TODO: Đối chiếu ngoại lệ IQR với dữ liệu gốc và lưu báo cáo ra PHASE2_TABLES_DIR / 'quality_and_iqr_report.csv'


### Trực quan hóa


In [ ]:
# TODO (T2.2c — Trực quan hóa)
# Vẽ heatmap/biểu đồ thanh tỷ lệ thiếu dữ liệu theo cột và boxplot (hoặc kết hợp stripplot/ECDF) thể hiện phân phối và các điểm ngoại lệ IQR trước khi kết luận.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T2.2c):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Tóm tắt các vấn đề thiếu dữ liệu và trùng lặp theo từng doanh nghiệp đại diện:** ...
- **Kết quả kiểm tra ngoại lệ IQR (ngưỡng, tỷ lệ vượt ngưỡng và giải thích thực tế từ dữ liệu gốc):** ...
- **Quyết định xử lý dữ liệu sạch (giữ nguyên / lọc dòng tổng cộng / gắn cờ) kèm bằng chứng:** ...


## Tác vụ T2.3a — Phân tích Khám phá nội tại từng doanh nghiệp đại diện

### Mục tiêu
Thực hiện phân tích mô tả chuyên sâu trên dữ liệu đã làm sạch (Hồi 2) để phát hiện mẫu hình vận hành, cơ cấu sản phẩm/dịch vụ, phân phối giá trị đơn hàng và các yếu tố đặc thù ngành của từng doanh nghiệp đại diện.

### Câu hỏi khám phá
- Top mặt hàng hoặc nhóm hàng nào đóng góp phần lớn doanh thu / số lượng (quy tắc tập trung kiểu Pareto)?
- Phân phối giá trị đơn hàng có đặc điểm gì (độ lệch phải, khoảng trung vị vs trung bình, phân hóa đơn lẻ và đơn lớn)?
- Các chiều đặc thù của từng doanh nghiệp (khách quen vs khách lẻ, hình thức thanh toán, ca sáng/chiều, lô thuốc gần hết hạn, công nợ, thợ sửa xe, máy cũ/trả góp...) thể hiện quy luật gì?

### Các bước thực hiện

**Bước 1 — Xác định bộ câu hỏi và chiều phân tích đặc thù cho từng doanh nghiệp đại diện**
Liệt kê các chiều dữ liệu khả dụng của từng doanh nghiệp đã chọn (bán hàng, nhóm hàng, khách hàng, thanh toán, tồn/lô/dịch vụ...).

**Bước 2 — Phân tích cơ cấu sản phẩm / nhóm hàng chủ lực**
Tính doanh thu, tỷ trọng % doanh thu, số lượng (đơn vị cơ bản) theo mặt hàng và nhóm hàng; áp dụng top-k + 'Khác' nếu số mặt hàng lớn.

**Bước 3 — Phân tích phân phối giá trị đơn hàng & cấu trúc giỏ hàng**
Tính các chỉ số thống kê mô tả bền vững (trung vị, Q25, Q75, p90, p95, trung bình, độ lệch chuẩn) cho giá trị đơn hàng và số dòng hàng/đơn.

**Bước 4 — Khám phá các chiều đặc thù theo ngành (khách hàng, thanh toán, ca, lô, dịch vụ...)**
Triển khai phân tích riêng cho đặc trưng của từng doanh nghiệp (tỷ trọng khách vãng lai, phương thức thanh toán, tỷ lệ hủy/trả, hoạt chất/hạn dùng, lượt bảo hành...).

**Bước 5 — Tổng hợp bảng bằng chứng thống kê và lưu `business_findings_summary.csv`**
Ghi nhận mọi con số kèm phạm vi ngày quan sát, cỡ mẫu `n`, nguồn bảng và lưu vào `PHASE2_TABLES_DIR / 'business_findings_summary.csv'`.

### Gợi ý phương pháp & Lưu ý
- Phân biệt rạch ròi đơn vị quan sát (*grain*): khi phân tích *giá trị đơn hàng*, phải gộp từ dòng hàng lên mã đơn trước rồi mới tính trung vị/histogram.
- Nếu giá trị đơn hàng trải qua nhiều bậc độ lớn (lệch phải mạnh), hãy vẽ thử cả thang tuyến tính và thang log (nhưng phải ghi chú rõ trên biểu đồ).
- Khi nhóm danh mục có quá nhiều giá trị (high cardinality), chỉ giữ Top 10–15 và gộp phần còn lại vào nhóm `Khác`.

### Triển khai


In [ ]:
# TODO (T2.3a — Bước 1: Xác định bộ câu hỏi và chiều phân tích đặc thù cho từng doanh nghiệp đại diện)
# TODO: Khai báo từ điển kế hoạch phân tích theo từng business_id đại diện


In [ ]:
# TODO (T2.3a — Bước 2: Phân tích cơ cấu sản phẩm / nhóm hàng chủ lực)
# TODO: Tính bảng xếp hạng và tỷ trọng sản phẩm/nhóm hàng cho từng doanh nghiệp đại diện


In [ ]:
# TODO (T2.3a — Bước 3: Phân tích phân phối giá trị đơn hàng & cấu trúc giỏ hàng)
# TODO: Tính thống kê phân phối giá trị đơn hàng (ở cấp đơn hàng, không nhầm với cấp dòng hàng)


In [ ]:
# TODO (T2.3a — Bước 4: Khám phá các chiều đặc thù theo ngành (khách hàng, thanh toán, ca, lô, dịch vụ...))
# TODO: Thực hiện phân tích đặc thù theo từng doanh nghiệp đại diện


In [ ]:
# TODO (T2.3a — Bước 5: Tổng hợp bảng bằng chứng thống kê và lưu `business_findings_summary.csv`)
# TODO: Tổng hợp bảng tóm tắt các phát hiện chính có dẫn chứng số liệu và lưu CSV vào PHASE2_TABLES_DIR


### Trực quan hóa


In [ ]:
# TODO (T2.3a — Trực quan hóa)
# Vẽ bộ biểu đồ khám phá đa dạng (histogram/ECDF giá trị đơn, thanh ngang top nhóm hàng, boxplot theo phương thức thanh toán/nhóm khách...) cho từng doanh nghiệp đại diện.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T2.3a):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Các phát hiện nổi bật theo từng doanh nghiệp đại diện (ghi rõ `business_id`, `n`, khoảng thời gian):** ...
- **Những phát hiện nào xứng đáng chọn làm điểm nhấn (1–2 biểu đồ/doanh nghiệp) để tinh chỉnh cho Hồi 2?:** ...


## Tác vụ T2.3b — Tinh chỉnh biểu đồ Giải thích Hồi 2

### Mục tiêu
Chọn lọc 1–2 phát hiện trọng tâm nhất cho mỗi doanh nghiệp đại diện và tinh chỉnh thành biểu đồ Giải thích đạt chuẩn Chương 2 §2.7 (Cleveland–McGill + 5 hướng dẫn Schwabish).

### Câu hỏi khám phá
- 1–2 thông điệp đắt giá nhất của mỗi doanh nghiệp đại diện đóng góp gì cho câu chuyện chung của đồ án?
- Loại biểu đồ được chọn (bar ngang, histogram+trung vị, boxplot/stripplot, stacked/100% bar) có phải kênh mã hóa chính xác nhất cho thông điệp đó không?
- Mỗi biểu đồ đã áp dụng tiêu đề chủ động, màu xám nền + màu nhấn, nhãn trực tiếp và chú thích nguồn/giới hạn chưa?

### Các bước thực hiện

**Bước 1 — Chọn lọc 1–2 phát hiện trọng tâm cho mỗi doanh nghiệp đại diện**
Xây dựng danh sách ngắn các biểu đồ Hồi 2 kèm câu chuyện/thông điệp một câu (active title) cho từng biểu đồ.

**Bước 2 — Chuẩn hóa dữ liệu vẽ (sắp xếp, gộp nhóm nhỏ, chuẩn hóa tỉ lệ)**
Sắp xếp các thanh theo giá trị giảm dần (trừ thứ tự tự nhiên), gộp phần đuôi nhỏ thành 'Khác', tính sẵn nhãn % hoặc giá trị.

**Bước 3 — Vẽ và tinh chỉnh biểu đồ theo 5 hướng dẫn Schwabish, lưu vào `PHASE2_FIGURES_DIR`**
Vẽ từng biểu đồ: bỏ viền trên/phải, nền xám cho đối tượng nền, tô màu nhấn (tối đa 1–2 màu), gắn nhãn trực tiếp, lưu vào `PHASE2_FIGURES_DIR`.

**Bước 4 — Ghi chú đầy đủ nguồn, đơn vị, cỡ mẫu và giới hạn**
Kiểm tra lại từng biểu đồ đảm bảo có đơn vị (VNĐ, %, đơn, lượt...), khoảng ngày quan sát, cỡ mẫu `n` và chú thích quy tắc `rulebook.md` áp dụng.

### Gợi ý phương pháp & Lưu ý
- Đặt tiêu đề chủ động nêu phát hiện (ví dụ: *'Nhóm X đóng góp Y% doanh thu nhưng chỉ chiếm Z% số đơn tại dev_00k'* thay vì *'Biểu đồ doanh thu dev_00k'*).
- Biểu đồ cột luôn bắt đầu từ gốc `0`; chú ý sử dụng đơn vị triệu/nghìn đồng cho trục dễ đọc thay vì số nguyên dài 8–9 chữ số.

### Triển khai


In [ ]:
# TODO (T2.3b — Bước 1: Chọn lọc 1–2 phát hiện trọng tâm cho mỗi doanh nghiệp đại diện)
# TODO: Liệt kê danh mục biểu đồ Giải thích Hồi 2 và tiêu đề chủ động dự kiến


In [ ]:
# TODO (T2.3b — Bước 2: Chuẩn hóa dữ liệu vẽ (sắp xếp, gộp nhóm nhỏ, chuẩn hóa tỉ lệ))
# TODO: Chuẩn bị DataFrame tóm tắt sạch cho từng biểu đồ Giải thích Hồi 2


In [ ]:
# TODO (T2.3b — Bước 3: Vẽ và tinh chỉnh biểu đồ theo 5 hướng dẫn Schwabish, lưu vào `PHASE2_FIGURES_DIR`)
# TODO: Vẽ hoàn chỉnh các biểu đồ Giải thích Hồi 2 và lưu vào PHASE2_FIGURES_DIR


In [ ]:
# TODO (T2.3b — Bước 4: Ghi chú đầy đủ nguồn, đơn vị, cỡ mẫu và giới hạn)
# TODO: Rà soát lại nhãn, đơn vị, chú thích để bàn giao sang NB04


### Trực quan hóa


In [ ]:
# TODO (T2.3b — Trực quan hóa)
# Trình bày hoàn chỉnh bộ biểu đồ Giải thích Hồi 2 cho từng doanh nghiệp đại diện.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T2.3b):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Tóm tắt thông điệp Hồi 2 (*Bằng chứng vi mô*) rút ra từ từng doanh nghiệp đại diện:** ...
- **Danh sách tệp biểu đồ Hồi 2 trong `PHASE2_FIGURES_DIR` và giới hạn đi kèm:** ...


## 05. Tổng kết giai đoạn & Bàn giao

> **TODO (Thành viên phụ trách Giai đoạn 2):** Hoàn thiện checklist trước khi bàn giao sang Giai đoạn 3 và 4.

- **Những công việc đã thực hiện:**
  - [ ] `T2.1`: Chốt danh sách tập doanh nghiệp đại diện -> `tables/representative_selection.csv`.
  - [ ] `T2.2a`: Xây dựng và kiểm chứng logic tính chỉ số theo `rulebook.md`.
  - [ ] `T2.2b`: Chuẩn hóa đơn vị cơ bản, giải mã `Ghi tắt`, xuất bảng Tidy -> `tidy_data/<business_id>_tidy.csv`.
  - [ ] `T2.2c`: Hoàn thành kiểm tra thiếu dữ liệu, trùng lặp, ngoại lệ IQR -> `tables/quality_and_iqr_report.csv`.
  - [ ] `T2.3a`–`T2.3b`: Khám phá nội tại (`tables/business_findings_summary.csv`) và tinh chỉnh bộ biểu đồ Hồi 2 (`figures/`).
- **Những phát hiện chính của Hồi 2:**
  - ...
- **Các vấn đề còn tồn tại / Khác biệt ngữ nghĩa phát hiện được:**
  - ...
- **Giới hạn của kết quả:**
  - ...
- **Đầu ra bàn giao cho Giai đoạn 3 (`03_cross_business_eda.ipynb`) và Giai đoạn 4 (`04_visualization_and_storytelling.ipynb`):**
  - `artifacts/guided_eda/phase2_business_eda/tidy_data/`
  - `artifacts/guided_eda/phase2_business_eda/tables/`
  - `artifacts/guided_eda/phase2_business_eda/figures/`
